# Font Settings by Cell Type (`window.craftFonts`)

> **Feature 4.1:** Per-cell-type font family, size, line height, and weight customization for **Prompt**, **Assistant response**, **Note**, **Code**, and **Raw** cells with live-rendered preview cards, instant live-reload on the active dialog, `localStorage` persistence, and **Copy as CRAFT.css snippet**.

## Lazy Loader (`CRAFT.js`)
Applies any persisted CSS font variables on startup and loads `/static/.craft-assets/craft_fonts.js` on first hotkey or API call.

In [ ]:
#| export_js CRAFT.js
#| eval: false
(() => {
  window.craftFonts?.disable?.();
  const CORE = window.__craftCMCore;
  if (!CORE || !CORE.features.init('font-settings', 'Font Settings', 'Per-cell-type font family, size, line-height, and weight')) return;
  const {store} = CORE;
  const STORE_KEY = 'craft-font-settings-v1';
  // Apply persisted font CSS variables immediately without loading modal code
  const saved = store.json(STORE_KEY, null);
  if (saved && typeof saved === 'object') {
    const root = document.documentElement.style;
    for (const k of ['prompt', 'response', 'note', 'code', 'raw']) {
      const c = saved[k];
      if (!c) continue;
      if (c.family) root.setProperty(`--craft-font-${k}-family`, c.family);
      if (c.size != null) root.setProperty(`--craft-font-${k}-size`, `${c.size}px`);
      if (c.lh != null) root.setProperty(`--craft-font-${k}-lh`, String(c.lh));
      if (c.weight != null) root.setProperty(`--craft-font-${k}-weight`, String(c.weight));
    }
    if (saved.code) {
      if (saved.code.size != null) root.setProperty('--craft-code', `${saved.code.size}px`);
      if (saved.code.lh != null) root.setProperty('--craft-code-lh', String(saved.code.lh));
      if (saved.code.family) root.setProperty('--craft-code-family', saved.code.family);
    }
  }
  let loading = null;
  const load = () => {
    if (window.craftFonts?.__loaded) return Promise.resolve(window.craftFonts);
    if (loading) return loading;
    loading = new Promise((resolve, reject) => {
      document.querySelector('script[data-craft-asset="craft_fonts.js"]')?.remove();
      const s = document.createElement('script');
      s.src = '/static/.craft-assets/craft_fonts.js';
      s.dataset.craftAsset = 'craft_fonts.js';
      s.onload = () => { loading = null; resolve(window.craftFonts); };
      s.onerror = err => { loading = null; reject(err); };
      document.head.append(s);
    });
    return loading;
  };
  window.__hotkeyRegister?.({id: 'craft-font-settings', label: 'Font settings (by cell type)', group: 'CRAFT · Appearance', combo: '', handler: () => { load().then(api => api?.open?.()); return true; }});
  window.__hotkeyRegister?.({id: 'craft-font-reset', label: 'Reset font settings to defaults', group: 'CRAFT · Appearance', combo: '', handler: () => { load().then(api => api?.reset?.()); return true; }});
  window.craftFonts = {
    open: () => load().then(api => api?.open?.()),
    reset: () => load().then(api => api?.reset?.()),
    get: () => load().then(api => api?.get?.()),
    apply: cfg => load().then(api => api?.apply?.(cfg)),
    snippet: () => load().then(api => api?.snippet?.()),
    disable() { delete window.craftFonts; },
  };
})();

## Templates (`#| export_ft`)

In [ ]:
#| export_ft craft_fonts.js
#| eval: false
_FONT_TABS = [
    ('prompt', 'Prompt'),
    ('response', 'Assistant response'),
    ('note', 'Note'),
    ('code', 'Code'),
    ('raw', 'Raw'),
]
_FAMILY_PRESETS = [
    ("'Craft Encode Sans', -apple-system, sans-serif", 'Craft Encode Sans (default prose)'),
    ("'Craft Iosevka', monospace", 'Craft Iosevka (default code)'),
    ("-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif", 'System Sans-Serif'),
    ("Georgia, 'Iowan Old Style', 'Times New Roman', serif", 'System Serif'),
    ("ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace", 'System Monospace'),
]

TEMPLATES = {
    'font_modal': craft_overlay(
        craft_card(
            Div(
                Strong('Font Settings by Cell Type'),
                Button('×', type='button', cls='craft-modal-x', aria_label='Close', data_ref='close'),
                cls='craft-modal-head',
            ),
            Div(
                Div(
                    *[Button(lbl, type='button', role='tab', cls='craft-font-tab', data_tab=key, aria_selected='false')
                      for key, lbl in _FONT_TABS],
                    cls='craft-font-tabs', role='tablist', data_ref='tabs',
                ),
                Div(
                    ft('datalist', *[Option(lbl, value=val) for val, lbl in _FAMILY_PRESETS], id='craft-font-presets'),
                    Div(
                        Label(
                            Span('CSS font-family'),
                            Input(cls='uk-input craft-font-ctrl', type='text', list='craft-font-presets',
                                  placeholder='Pick a preset or type a CSS font-family stack', data_ref='family'),
                            cls='craft-font-row',
                        ),
                        Label(
                            Span('Size ', Span(cls='craft-font-val', data_ref='sizeVal')),
                            Input(type='range', min='8', max='24', step='0.5', cls='craft-font-range', data_ref='size'),
                            cls='craft-font-row',
                        ),
                        Label(
                            Span('Line height ', Span(cls='craft-font-val', data_ref='lhVal')),
                            Input(type='range', min='1.0', max='2.0', step='0.05', cls='craft-font-range', data_ref='lh'),
                            cls='craft-font-row',
                        ),
                        Label(
                            Span('Weight ', Span(cls='craft-font-val', data_ref='wtVal')),
                            Input(type='range', min='200', max='700', step='20', cls='craft-font-range', data_ref='weight'),
                            cls='craft-font-row',
                        ),
                        cls='craft-font-grid',
                    ),
                    Div(
                        Div(cls='craft-font-prev-lbl', data_ref='prevLbl'),
                        Div(cls='craft-font-preview', data_ref='preview'),
                        cls='craft-font-prev-wrap',
                    ),
                    cls='craft-font-body', data_ref='body',
                ),
                cls='craft-font-main',
            ),
            Div(
                craft_btn('Copy as CRAFT.css snippet', size='xs', data_ref='copyCss'),
                craft_btn('Reset to defaults', size='xs', data_ref='reset'),
                craft_btn('Save & Close', variant='primary', size='xs', data_ref='save'),
                cls='craft-modal-foot',
            ),
            cls='craft-font-card',
        ),
        id='craft-font-modal', aria_label='Font Settings by Cell Type',
    ),
}

## Typography Defaults, Persistence & CSS Application (`#| export_js_iife`)


In [ ]:
#| export_js_iife craft_fonts.js craftFonts
#| eval: false
window.craftFonts?.disable?.();

const CORE = window.__craftCMCore;
if (!CORE || !CORE.features.isEnabled('font-settings')) return;
const {on, store, mountModal, notify} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal};
const STORE_KEY = 'craft-font-settings-v1';

const TYPES = [
  {key: 'prompt',   label: 'Prompt',             defFamily: "'Craft Encode Sans', -apple-system, sans-serif", defSize: 11, defLh: 1.2, defWeight: 360, sample: 'You: Summarize the trade-offs of this indexing strategy in 3 bullet points.'},
  {key: 'response', label: 'Assistant response',  defFamily: "'Craft Encode Sans', -apple-system, sans-serif", defSize: 11, defLh: 1.25, defWeight: 360, sample: 'Assistant: Covering indexes avoid heap lookups, at the cost of extra write amplification and storage.'},
  {key: 'note',     label: 'Note',               defFamily: "'Craft Encode Sans', -apple-system, sans-serif", defSize: 11, defLh: 1.25, defWeight: 360, sample: 'Design note — keep per-cell typography variables falling back to --craft-prose and --craft-code.'},
  {key: 'code',     label: 'Code',               defFamily: "'Craft Iosevka', monospace",                     defSize: 10, defLh: 1.2,  defWeight: 400, sample: 'def fib(n: int) -> int:\n    return n if n < 2 else fib(n - 1) + fib(n - 2)'},
  {key: 'raw',      label: 'Raw',                defFamily: "'Craft Iosevka', monospace",                     defSize: 10, defLh: 1.25, defWeight: 400, sample: '{"status": "ok", "tokens": 1257, "mode": "raw"}'},
];

const defaults = () => Object.fromEntries(TYPES.map(t => [t.key, {
  family: t.defFamily, size: t.defSize, lh: t.defLh, weight: t.defWeight,
}]));

const loadSettings = () => {
  const saved = store.json(STORE_KEY, null);
  const base = defaults();
  if (!saved || typeof saved !== 'object') return base;
  for (const t of TYPES) {
    if (saved[t.key]) base[t.key] = {...base[t.key], ...saved[t.key]};
  }
  return base;
};

let current = loadSettings();
let modal = null, activeTab = 'prompt';

function applySettings(cfg = current) {
  const root = document.documentElement.style;
  for (const t of TYPES) {
    const c = cfg[t.key];
    if (!c) continue;
    root.setProperty(`--craft-font-${t.key}-family`, c.family);
    root.setProperty(`--craft-font-${t.key}-size`, `${c.size}px`);
    root.setProperty(`--craft-font-${t.key}-lh`, String(c.lh));
    root.setProperty(`--craft-font-${t.key}-weight`, String(c.weight));
  }
  if (cfg.code) {
    root.setProperty('--craft-code', `${cfg.code.size}px`);
    root.setProperty('--craft-code-lh', String(cfg.code.lh));
    root.setProperty('--craft-code-family', cfg.code.family);
  }
}

function buildCssSnippet(cfg = current) {
  const lines = [':root {'];
  for (const t of TYPES) {
    const c = cfg[t.key];
    lines.push(`  --craft-font-${t.key}-family: ${c.family};`);
    lines.push(`  --craft-font-${t.key}-size: ${c.size}px;`);
    lines.push(`  --craft-font-${t.key}-lh: ${c.lh};`);
    lines.push(`  --craft-font-${t.key}-weight: ${c.weight};`);
  }
  lines.push('}');
  return lines.join('\n');
}


## Font Settings Modal, Hotkeys & Public API (`#| export_js_iife`)

In [ ]:
#| export_js_iife craft_fonts.js craftFonts
#| eval: false
const activeType = () => TYPES.find(x => x.key === activeTab) || TYPES[0];

function renderModal() {
  if (!modal) return;
  for (const btn of modal.tabs.querySelectorAll('[data-tab]')) {
    const active = btn.dataset.tab === activeTab;
    btn.classList.toggle('is-active', active);
    btn.setAttribute('aria-selected', String(active));
  }
  const t = activeType(), cfg = current[t.key];
  modal.family.value = cfg.family;
  modal.size.value = String(cfg.size);
  modal.sizeVal.textContent = `${cfg.size}px`;
  modal.lh.value = String(cfg.lh);
  modal.lhVal.textContent = String(cfg.lh);
  modal.weight.value = String(cfg.weight);
  modal.wtVal.textContent = String(cfg.weight);
  modal.prevLbl.textContent = `Live preview — ${t.label}`;
  modal.preview.className = `craft-font-preview is-${t.key}`;
  modal.preview.textContent = t.sample;
  applySettings(current);
}

let revertOnClose = true;

function openModal() {
  if (!modal) {
    modal = mountModal('font_modal', {
      refs: true,
      signal: ac.signal,
      onClose: () => {
        if (revertOnClose) { current = loadSettings(); applySettings(current); }
        revertOnClose = true;
      },
    });
    on(modal.tabs, 'click', e => {
      const btn = e.target.closest('[data-tab]');
      if (!btn) return;
      activeTab = btn.dataset.tab;
      renderModal();
    }, sig);
    on(modal.family, 'input', () => {
      const t = activeType();
      current[t.key].family = modal.family.value.trim() || t.defFamily;
      applySettings(current);
    }, sig);
    on(modal.size, 'input', () => {
      const v = parseFloat(modal.size.value);
      current[activeType().key].size = v;
      modal.sizeVal.textContent = `${v}px`;
      applySettings(current);
    }, sig);
    on(modal.lh, 'input', () => {
      const v = parseFloat(modal.lh.value);
      current[activeType().key].lh = v;
      modal.lhVal.textContent = String(v);
      applySettings(current);
    }, sig);
    on(modal.weight, 'input', () => {
      const v = parseInt(modal.weight.value, 10);
      current[activeType().key].weight = v;
      modal.wtVal.textContent = String(v);
      applySettings(current);
    }, sig);
    on(modal.save, 'click', () => {
      store.setJson(STORE_KEY, current);
      applySettings(current);
      revertOnClose = false;
      modal.close();
      notify('Font settings saved.', 'success');
    }, sig);
    on(modal.reset, 'click', () => {
      current = defaults();
      store.del(STORE_KEY);
      applySettings(current);
      renderModal();
      notify('Font settings reset to defaults.');
    }, sig);
    on(modal.copyCss, 'click', async () => {
      const css = buildCssSnippet(current);
      try {
        await navigator.clipboard.writeText(css);
        notify('Copied CRAFT.css font snippet to clipboard.', 'success');
      } catch {
        prompt('Copy CRAFT.css snippet:', css);
      }
    }, sig);
  }
  current = loadSettings();
  revertOnClose = true;
  renderModal();
  modal.open();
}

// Apply persisted settings immediately on load
if (store.json(STORE_KEY, null)) applySettings(current);

window.__hotkeyRegister?.({id: 'craft-font-settings', label: 'Font settings (by cell type)', group: 'CRAFT · Appearance', combo: '', handler: () => { openModal(); return true; }});
window.__hotkeyRegister?.({id: 'craft-font-reset', label: 'Reset font settings to defaults', group: 'CRAFT · Appearance', combo: '', handler: () => {
  current = defaults(); store.del(STORE_KEY); applySettings(current);
  notify('Font settings reset to defaults.');
  return true;
}});

window.craftFonts = {
  __loaded: true,
  open: openModal,
  get: () => loadSettings(),
  apply: applySettings,
  reset: () => { current = defaults(); store.del(STORE_KEY); applySettings(current); },
  snippet: () => buildCssSnippet(current),
  disable() {
    ac.abort();
    modal?.destroy?.();
    delete window.craftFonts;
  },
};

## Font Settings CSS (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Font Settings Modal (14_font_settings) ────────────────────────── */
.craft-font-card { width: min(620px, 92vw); max-height: 80vh; display: flex; flex-direction: column; overflow: hidden; }
.craft-font-main { display: flex; flex-direction: column; overflow-y: auto; }
.craft-font-tabs { display: flex; flex-wrap: wrap; gap: 4px; padding: 10px 14px 0; border-bottom: 1px solid var(--craft-border); }
.craft-font-tab {
  padding: 6px 10px; border: 0; border-bottom: 2px solid transparent; background: transparent;
  color: var(--craft-muted); font-size: 11.5px; cursor: pointer;
}
.craft-font-tab:hover { color: var(--craft-fg-strong); }
.craft-font-tab.is-active { color: var(--craft-accent); border-bottom-color: var(--craft-accent); font-weight: 600; }
.craft-font-body { padding: 14px 16px; display: flex; flex-direction: column; gap: 14px; }
.craft-font-grid { display: flex; flex-direction: column; gap: 10px; }
.craft-font-row { display: grid; grid-template-columns: 130px 1fr; align-items: center; gap: 10px; font-size: 11.5px; color: var(--craft-fg); }
.craft-font-val { font-family: var(--craft-code-family); color: var(--craft-accent); margin-left: 4px; }
.craft-font-ctrl { height: 28px !important; font-size: 11.5px !important; }
.craft-font-range { width: 100%; accent-color: var(--craft-accent); }
.craft-font-prev-wrap { padding: 10px 12px; border-radius: 6px; background: var(--craft-bg); border: 1px solid var(--craft-border); }
.craft-font-prev-lbl { font-size: 10px; text-transform: uppercase; letter-spacing: .04em; color: var(--craft-meta-fg, var(--craft-muted)); margin-bottom: 6px; }
.craft-font-preview { color: var(--craft-fg); white-space: pre-wrap; }
.craft-font-preview.is-prompt   { font-family: var(--craft-font-prompt-family);   font-size: var(--craft-font-prompt-size);   line-height: var(--craft-font-prompt-lh);   font-weight: var(--craft-font-prompt-weight); }
.craft-font-preview.is-response { font-family: var(--craft-font-response-family); font-size: var(--craft-font-response-size); line-height: var(--craft-font-response-lh); font-weight: var(--craft-font-response-weight); }
.craft-font-preview.is-note     { font-family: var(--craft-font-note-family);     font-size: var(--craft-font-note-size);     line-height: var(--craft-font-note-lh);     font-weight: var(--craft-font-note-weight); }
.craft-font-preview.is-code     { font-family: var(--craft-font-code-family);     font-size: var(--craft-font-code-size);     line-height: var(--craft-font-code-lh);     font-weight: var(--craft-font-code-weight);     background: var(--craft-recessed); padding: 8px 10px; border-radius: 4px; }
.craft-font-preview.is-raw      { font-family: var(--craft-font-raw-family);      font-size: var(--craft-font-raw-size);      line-height: var(--craft-font-raw-lh);      font-weight: var(--craft-font-raw-weight);      background: var(--craft-recessed); padding: 8px 10px; border-radius: 4px; }

Verify `14_font_settings.ipynb` exports `window.craftFonts`, its template, and CSS.

In [ ]:
import os, subprocess
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/14_font_settings.ipynb')
stub = '\n'.join(code for _, code in assets['CRAFT.js'])
js = '\n'.join(code for _, code in assets['craft_fonts.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
assert 'craft_fonts.js' in stub and 'font-settings' in stub
for sym in ('window.craftFonts', 'font_modal', 'buildCssSnippet'):
    assert sym in js, f"Missing {sym}"
assert '.craft-font-card' in css and '.craft-font-preview' in css

node_script = r'''
const assert = require('assert');
const src = process.env.FONT_JS;
const i0 = src.indexOf('const TYPES =');
const i1 = src.indexOf('const loadSettings =');
const j0 = src.indexOf('function buildCssSnippet');
const j1 = src.indexOf('function renderModal');
const {defaults, buildCssSnippet} = new Function(
  src.slice(i0, i1) + '\n' + src.slice(j0, j1) + '\nreturn {defaults, buildCssSnippet};'
)();
const cfg = defaults();
const out = buildCssSnippet(cfg);
assert(out.startsWith(':root {') && out.endsWith('}'));
for (const k of ['prompt', 'response', 'note', 'code', 'raw']) {
  assert(out.includes(`--craft-font-${k}-family:`), `missing ${k} family`);
  assert(out.includes(`--craft-font-${k}-size:`), `missing ${k} size`);
}
'''
subprocess.run(['node', '-e', node_script], env={**os.environ, 'FONT_JS': js}, check=True)
